# AquaRede · CNN de imagens de satélite

Global Solution 2026.2 · Software Engineering (Sprint 1, cartão **[IA/CNN] Preparar e adaptar a CNN**)

**Objetivo:** classificar imagens de satélite em quatro classes úteis para a AquaRede e devolver *classe + confiança*, no formato da camada `satelite` do contrato JSON do alerta.

| Classe AquaRede | Classes originais do EuroSAT |
|---|---|
| `agua` | River, SeaLake |
| `urbano` | Highway, Industrial, Residential |
| `vegetacao_densa` | Forest, PermanentCrop |
| `campo_aberto` | AnnualCrop, HerbaceousVegetation, Pasture |

> **Limitação (declarar na apresentação):** o EuroSAT não tem uma classe "solo seco". A CNN identifica **água** e **cobertura do solo**; o **clima seco** vem da API OpenWeather (dias sem chuva). `campo_aberto` é só um indicador de área mais exposta.

In [ ]:
# O servidor original do EuroSAT (madm.dfki.de) está fora do ar; usamos um espelho no Hugging Face
!pip -q install datasets

In [ ]:
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Dataset (EuroSAT RGB, via Hugging Face) e divisão treino / validação / teste

In [ ]:
from datasets import load_dataset

# espelho do EuroSAT RGB no Hugging Face (o primeiro que funcionar é usado)
for repo in ['giswqs/EuroSAT_RGB', 'blanchon/EuroSAT_RGB']:
    try:
        dsd = load_dataset(repo)
        print('Dataset carregado de:', repo)
        break
    except Exception as e:
        print('Falhou', repo, '->', type(e).__name__)
else:
    raise RuntimeError('Nenhum espelho do EuroSAT funcionou. Veja a mensagem de erro acima.')

# se o espelho não trouxer validação/teste, divide 70/15/15 com seed fixa
if not all(s in dsd for s in ['train', 'validation', 'test']):
    todo = dsd['train']
    parte = todo.train_test_split(test_size=0.30, seed=42, stratify_by_column='label')
    resto = parte['test'].train_test_split(test_size=0.50, seed=42, stratify_by_column='label')
    dsd = {'train': parte['train'], 'validation': resto['train'], 'test': resto['test']}

def normalizar(nome):
    nome = nome.replace(' ', '').replace('_', '')
    return {'IndustrialBuildings': 'Industrial', 'ResidentialBuildings': 'Residential'}.get(nome, nome)

NOMES_ORIGINAIS = [normalizar(n) for n in dsd['train'].features['label'].names]
print('Classes originais:', NOMES_ORIGINAIS)

def para_numpy(split):
    d = dsd[split]
    X = np.stack([np.asarray(im.convert('RGB').resize((64, 64))) for im in d['image']])
    y = np.array(d['label'], dtype=np.int32)
    return X, y

X_tr, y_tr = para_numpy('train')
X_va, y_va = para_numpy('validation')
X_te, y_te = para_numpy('test')
print('Treino:', len(X_tr), '| Validação:', len(X_va), '| Teste:', len(X_te))

ds_train = tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
ds_val = tf.data.Dataset.from_tensor_slices((X_va, y_va))
ds_test = tf.data.Dataset.from_tensor_slices((X_te, y_te))

## 2. Reagrupar as classes para o objetivo da AquaRede

In [ ]:
CLASSES = ['agua', 'urbano', 'vegetacao_densa', 'campo_aberto']
MAPA = {
    'River': 'agua', 'SeaLake': 'agua',
    'Highway': 'urbano', 'Industrial': 'urbano', 'Residential': 'urbano',
    'Forest': 'vegetacao_densa', 'PermanentCrop': 'vegetacao_densa',
    'AnnualCrop': 'campo_aberto', 'HerbaceousVegetation': 'campo_aberto', 'Pasture': 'campo_aberto',
}
lut = tf.constant([CLASSES.index(MAPA[n]) for n in NOMES_ORIGINAIS], dtype=tf.int32)

BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE

def preparar(img, label):
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.gather(lut, label)

train = ds_train.map(preparar, AUTOTUNE).shuffle(2000, seed=42).batch(BATCH).prefetch(AUTOTUNE)
val = ds_val.map(preparar, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
test = ds_test.map(preparar, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

# distribuição das classes no treino (para checar desbalanceamento)
y_train = np.concatenate([y.numpy() for _, y in train])
for i, c in enumerate(CLASSES):
    print(f'{c:16s} {np.sum(y_train == i):6d} imagens')

pesos = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y_train)
class_weight = {i: float(p) for i, p in enumerate(pesos)}
print('Pesos das classes:', class_weight)

In [ ]:
# amostras para conferir o reagrupamento
imgs, labels = next(iter(train))
plt.figure(figsize=(12, 3))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(imgs[i].numpy())
    plt.title(CLASSES[int(labels[i])], fontsize=8)
    plt.axis('off')
plt.show()

## 3. Modelo (CNN própria com data augmentation)

In [ ]:
aug = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='augmentation')

def bloco(filtros):
    return [
        layers.Conv2D(filtros, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
    ]

model = models.Sequential([
    layers.Input(shape=(64, 64, 3)),
    aug,
    *bloco(32), *bloco(64), *bloco(128),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 4. Treinamento

In [ ]:
parar_cedo = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

historico = model.fit(train, validation_data=val, epochs=20,
                      class_weight=class_weight, callbacks=[parar_cedo])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(historico.history['accuracy'], label='treino')
ax[0].plot(historico.history['val_accuracy'], label='validação')
ax[0].set_title('Acurácia'); ax[0].legend()
ax[1].plot(historico.history['loss'], label='treino')
ax[1].plot(historico.history['val_loss'], label='validação')
ax[1].set_title('Perda'); ax[1].legend()
plt.show()

## 5. Avaliação no conjunto de teste (evidência para o cartão de validação)

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test])
y_prob = model.predict(test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues', xticks_rotation=30)
plt.show()

## 6. Exportar o modelo

In [ ]:
model.save('aquarede_cnn.keras')
with open('classes.json', 'w', encoding='utf-8') as f:
    json.dump(CLASSES, f, ensure_ascii=False)
print('Salvos: aquarede_cnn.keras e classes.json')

# no Colab, para baixar os arquivos:
# from google.colab import files
# files.download('aquarede_cnn.keras'); files.download('classes.json')

## 7. Função de inferência: imagem → classe + confiança

A saída `camada_satelite` segue o contrato JSON do alerta (`camada`, `evidencia`, `peso`).

In [ ]:
from PIL import Image

def prever(imagem):
    """Recebe um caminho de arquivo, uma imagem PIL ou um array numpy."""
    if isinstance(imagem, str):
        imagem = Image.open(imagem)
    elif isinstance(imagem, np.ndarray):
        imagem = Image.fromarray(imagem.astype('uint8'))
    x = np.asarray(imagem.convert('RGB').resize((64, 64)), dtype='float32') / 255.0
    p = model.predict(x[None, ...], verbose=0)[0]
    i = int(np.argmax(p))
    return {
        'classe': CLASSES[i],
        'confianca': round(float(p[i]), 3),
        'probabilidades': {c: round(float(v), 3) for c, v in zip(CLASSES, p)},
    }

DESCRICAO = {
    'agua': 'presença de corpo d\'água',
    'urbano': 'uso urbano denso',
    'vegetacao_densa': 'vegetação densa',
    'campo_aberto': 'campo aberto, mais exposto',
}

# TODO: valores iniciais, ajustar e justificar com o grupo (0 = menor contribuição ao risco, 1 = maior)
CONTRIBUICAO_RISCO = {'agua': 0.2, 'urbano': 0.8, 'vegetacao_densa': 0.3, 'campo_aberto': 0.6}

def camada_satelite(resultado, peso=0.15):
    """Devolve (item de 'motivos' do alerta, valor normalizado da camada para o score)."""
    motivo = {
        'camada': 'satelite',
        'evidencia': f"CNN classificou a área como {DESCRICAO[resultado['classe']]} (confiança {resultado['confianca']:.2f})",
        'peso': peso,
    }
    return motivo, CONTRIBUICAO_RISCO[resultado['classe']]

# teste rápido com uma imagem do conjunto de teste
img_teste, lab_teste = next(iter(ds_test.take(1)))
r = prever(img_teste.numpy())
print('Real:', CLASSES[int(lut[lab_teste])], '| Previsto:', r)
print(camada_satelite(r))

## 8. Demo com Gradio (opcional)

In [ ]:
!pip -q install gradio
import gradio as gr

def classificar(img):
    return prever(img)['probabilidades']

gr.Interface(fn=classificar,
             inputs=gr.Image(type='pil', label='Imagem de satélite'),
             outputs=gr.Label(num_top_classes=4, label='Classe prevista'),
             title='AquaRede · CNN de satélite').launch()